# 05b — Complemento da validação do módulo de risco

Este notebook complementa a validação realizada no `05_validacao_risco.ipynb`.

Na primeira validação foram selecionados 50 cenários, mas 16 precisaram ser excluídos porque:

- a classe detectada estava incorreta;
- a bounding box não representava adequadamente o objeto;
- o cenário era ambíguo para avaliação de risco.

Restaram **34 cenários válidos**.

## Objetivo desta etapa

Buscar novos cenários apenas para substituir os 16 excluídos:

```text
bed       → 3 novos válidos
sofa      → 5 novos válidos
wardrobe  → 8 novos válidos
```

Como alguns novos casos ainda podem precisar ser excluídos após inspeção visual,
este notebook gera uma quantidade maior de candidatos:

```text
bed       → até 6 candidatos
sofa      → até 10 candidatos
wardrobe  → até 16 candidatos
```

## Diferença em relação à primeira seleção

Desta vez não utilizaremos apenas a previsão do YOLO.

Um candidato só será aceito quando:

```text
classe prevista == classe do ground truth
+
confiança >= 0,25
+
IoU >= 0,50
```

Isso separa melhor a avaliação do detector da avaliação do módulo de risco.

> Este notebook NÃO altera as regras V1 ou V2.

## 1. Resultado final esperado

Ao final desta primeira fase serão criados:

```text
reports/
├── validation_risk/
│   └── candidatos_complementares/
│       ├── R001_bed.jpg
│       ├── R002_bed.jpg
│       ├── ...
│       └── Rxxx_wardrobe.jpg
│
└── tables/
    └── candidatos_complementares_risco.csv
```

O CSV terá a coluna `risco_esperado` vazia para preenchimento manual posterior.

As previsões das regras V1 e V2 **não são exibidas**, evitando influenciar a rotulagem humana.

## 2. Importações e localização do projeto

In [ ]:
from pathlib import Path
import random
import sys

import pandas as pd
import yaml
from PIL import Image, ImageDraw


def localizar_raiz_projeto() -> Path:
    atual = Path.cwd().resolve()

    for candidata in [atual, *atual.parents]:
        if (
            (candidata / "requirements.txt").exists()
            and (candidata / "src").exists()
        ):
            return candidata

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto."
    )


RAIZ_PROJETO = localizar_raiz_projeto()

if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

print("Raiz do projeto:", RAIZ_PROJETO)

## 3. Importar Ultralytics

In [ ]:
try:
    from ultralytics import YOLO
except ImportError:
    import subprocess

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "ultralytics"]
    )

    from ultralytics import YOLO

print("Ultralytics importado com sucesso.")

## 4. Caminhos e parâmetros

In [ ]:
CAMINHO_MODELO = (
    RAIZ_PROJETO
    / "models"
    / "yolov8n_baseline"
    / "best.pt"
)

PASTA_DATASET = (
    RAIZ_PROJETO
    / "data"
    / "processed"
    / "homeobjects_yolo"
)

PASTA_IMAGENS_TEST = PASTA_DATASET / "images" / "test"
PASTA_LABELS_TEST = PASTA_DATASET / "labels" / "test"
CAMINHO_DATA_YAML = PASTA_DATASET / "data.yaml"

CAMINHO_GABARITO_ANTERIOR = (
    RAIZ_PROJETO
    / "reports"
    / "tables"
    / "validacao_manual_risco.csv"
)

PASTA_CANDIDATOS = (
    RAIZ_PROJETO
    / "reports"
    / "validation_risk"
    / "candidatos_complementares"
)

PASTA_TABELAS = (
    RAIZ_PROJETO
    / "reports"
    / "tables"
)

CAMINHO_CSV_CANDIDATOS = (
    PASTA_TABELAS
    / "candidatos_complementares_risco.csv"
)

CONFIANCA_MINIMA = 0.25
IOU_MINIMO = 0.50
IMGSZ = 640
SEMENTE = 20260913

# Quantidade realmente necessária para completar os 50 válidos.
NECESSARIOS = {
    "bed": 3,
    "sofa": 5,
    "wardrobe": 8,
}

# Geramos o dobro para termos opções após a revisão visual.
META_CANDIDATOS = {
    classe: quantidade * 2
    for classe, quantidade in NECESSARIOS.items()
}

PASTA_CANDIDATOS.mkdir(parents=True, exist_ok=True)
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)

caminhos_obrigatorios = [
    CAMINHO_MODELO,
    PASTA_IMAGENS_TEST,
    PASTA_LABELS_TEST,
    CAMINHO_DATA_YAML,
    CAMINHO_GABARITO_ANTERIOR,
]

for caminho in caminhos_obrigatorios:
    if not caminho.exists():
        raise FileNotFoundError(
            f"Caminho obrigatório não encontrado:\n{caminho}"
        )

print("Modelo:", CAMINHO_MODELO)
print("Dataset:", PASTA_DATASET)
print("Gabarito anterior:", CAMINHO_GABARITO_ANTERIOR)
print()
print("Necessários:", NECESSARIOS)
print("Meta de candidatos:", META_CANDIDATOS)

## 5. Ler as classes diretamente do `data.yaml`

O notebook não assume manualmente os IDs das classes.

Isso evita erros caso a ordem das classes no dataset seja diferente.

In [ ]:
with open(
    CAMINHO_DATA_YAML,
    "r",
    encoding="utf-8",
) as arquivo:
    configuracao_dataset = yaml.safe_load(arquivo)

nomes_yaml = configuracao_dataset["names"]

if isinstance(nomes_yaml, dict):
    NOMES_CLASSES = {
        int(chave): valor
        for chave, valor in nomes_yaml.items()
    }
else:
    NOMES_CLASSES = {
        indice: nome
        for indice, nome in enumerate(nomes_yaml)
    }

ID_POR_CLASSE = {
    nome: indice
    for indice, nome in NOMES_CLASSES.items()
}

print("Classes encontradas:")
for indice, nome in NOMES_CLASSES.items():
    print(f"{indice}: {nome}")

for classe in NECESSARIOS:
    if classe not in ID_POR_CLASSE:
        raise ValueError(
            f"A classe '{classe}' não existe no data.yaml."
        )

# 6. Identificar imagens já utilizadas

Todas as imagens que participaram da primeira seleção de 50 cenários
serão removidas desta nova busca.

Isso garante que os candidatos complementares sejam realmente novos.

In [ ]:
df_gabarito_anterior = pd.read_csv(
    CAMINHO_GABARITO_ANTERIOR,
    encoding="utf-8-sig",
)

if "arquivo" not in df_gabarito_anterior.columns:
    raise ValueError(
        "O CSV anterior não possui a coluna 'arquivo'."
    )

imagens_ja_utilizadas = set(
    df_gabarito_anterior["arquivo"]
    .dropna()
    .astype(str)
    .tolist()
)

print(
    "Imagens já utilizadas na primeira validação:",
    len(imagens_ja_utilizadas),
)

print(
    "Linhas do gabarito anterior:",
    len(df_gabarito_anterior),
)

# 7. Funções auxiliares

O ground truth está no formato YOLO:

```text
classe x_centro y_centro largura altura
```

com coordenadas normalizadas entre 0 e 1.

Vamos converter para:

```text
x1, y1, x2, y2
```

em pixels e calcular IoU com cada detecção prevista.

In [ ]:
def yolo_para_xyxy(
    x_centro,
    y_centro,
    largura_box,
    altura_box,
    largura_imagem,
    altura_imagem,
):
    x_centro *= largura_imagem
    y_centro *= altura_imagem
    largura_box *= largura_imagem
    altura_box *= altura_imagem

    x1 = x_centro - largura_box / 2
    y1 = y_centro - altura_box / 2
    x2 = x_centro + largura_box / 2
    y2 = y_centro + altura_box / 2

    return x1, y1, x2, y2


def calcular_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    largura_inter = max(0, inter_x2 - inter_x1)
    altura_inter = max(0, inter_y2 - inter_y1)

    area_inter = largura_inter * altura_inter

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)

    area_uniao = area_a + area_b - area_inter

    if area_uniao <= 0:
        return 0.0

    return area_inter / area_uniao


def ler_ground_truth(
    caminho_label,
    largura_imagem,
    altura_imagem,
):
    anotacoes = []

    if not caminho_label.exists():
        return anotacoes

    conteudo = caminho_label.read_text(
        encoding="utf-8"
    ).strip()

    if not conteudo:
        return anotacoes

    for indice_gt, linha in enumerate(
        conteudo.splitlines()
    ):
        partes = linha.strip().split()

        if len(partes) < 5:
            continue

        classe_id = int(float(partes[0]))

        if classe_id not in NOMES_CLASSES:
            continue

        classe = NOMES_CLASSES[classe_id]

        x_centro = float(partes[1])
        y_centro = float(partes[2])
        largura_box = float(partes[3])
        altura_box = float(partes[4])

        xyxy = yolo_para_xyxy(
            x_centro=x_centro,
            y_centro=y_centro,
            largura_box=largura_box,
            altura_box=altura_box,
            largura_imagem=largura_imagem,
            altura_imagem=altura_imagem,
        )

        anotacoes.append(
            {
                "gt_id": indice_gt,
                "classe_id": classe_id,
                "classe": classe,
                "xyxy": xyxy,
            }
        )

    return anotacoes

## 8. Funções espaciais

Estas informações serão incluídas no CSV apenas para documentação dos candidatos.

Elas **não serão utilizadas para escolher se o candidato entra ou não**.

In [ ]:
LIMITE_ESQUERDA = 1 / 3
LIMITE_DIREITA = 2 / 3

LIMITE_AREA_MEDIA = 0.05
LIMITE_AREA_GRANDE = 0.20

LIMITE_VERTICAL_SUPERIOR = 0.60
LIMITE_VERTICAL_INFERIOR = 0.80


def classificar_posicao(
    centro_x,
    largura_imagem,
):
    proporcao = centro_x / largura_imagem

    if proporcao < LIMITE_ESQUERDA:
        return "esquerda"

    if proporcao < LIMITE_DIREITA:
        return "centro"

    return "direita"


def classificar_tamanho(
    area_box,
    area_imagem,
):
    proporcao = area_box / area_imagem

    if proporcao < LIMITE_AREA_MEDIA:
        return "pequeno", proporcao

    if proporcao < LIMITE_AREA_GRANDE:
        return "medio", proporcao

    return "grande", proporcao


def classificar_vertical(
    y2,
    altura_imagem,
):
    proporcao = y2 / altura_imagem

    if proporcao < LIMITE_VERTICAL_SUPERIOR:
        return "superior", proporcao

    if proporcao < LIMITE_VERTICAL_INFERIOR:
        return "intermediaria", proporcao

    return "inferior", proporcao

## 9. Carregar o modelo

In [ ]:
modelo = YOLO(str(CAMINHO_MODELO))

print("Modelo carregado com sucesso.")
print(modelo.names)

# 10. Localizar todas as imagens TEST ainda não utilizadas

In [ ]:
EXTENSOES_IMAGEM = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

todas_imagens_test = sorted(
    arquivo
    for arquivo in PASTA_IMAGENS_TEST.rglob("*")
    if arquivo.is_file()
    and arquivo.suffix.lower() in EXTENSOES_IMAGEM
)

imagens_disponiveis = [
    arquivo
    for arquivo in todas_imagens_test
    if arquivo.name not in imagens_ja_utilizadas
]

print("Total de imagens TEST:", len(todas_imagens_test))
print(
    "Imagens removidas por já terem sido usadas:",
    len(todas_imagens_test) - len(imagens_disponiveis),
)
print("Imagens disponíveis:", len(imagens_disponiveis))

# 11. Buscar detecções corretas

Para cada imagem ainda não utilizada:

1. lemos o ground truth;
2. verificamos se existe `bed`, `sofa` ou `wardrobe`;
3. executamos o YOLO;
4. procuramos previsões da mesma classe;
5. calculamos IoU;
6. mantemos apenas correspondências com:

```text
confiança >= 0,25
IoU >= 0,50
```

Para evitar duplicação artificial, cada ground truth e cada previsão
só podem participar de uma correspondência.

In [ ]:
candidatos_validos = []

for indice_imagem, caminho_imagem in enumerate(
    imagens_disponiveis,
    start=1,
):
    if (
        indice_imagem == 1
        or indice_imagem % 25 == 0
        or indice_imagem == len(imagens_disponiveis)
    ):
        print(
            f"[{indice_imagem:03d}/{len(imagens_disponiveis)}] "
            f"{caminho_imagem.name}"
        )

    imagem = Image.open(caminho_imagem).convert("RGB")
    largura_imagem, altura_imagem = imagem.size
    area_imagem = largura_imagem * altura_imagem

    caminho_label = (
        PASTA_LABELS_TEST
        / f"{caminho_imagem.stem}.txt"
    )

    ground_truth = ler_ground_truth(
        caminho_label=caminho_label,
        largura_imagem=largura_imagem,
        altura_imagem=altura_imagem,
    )

    gt_alvo = [
        item
        for item in ground_truth
        if item["classe"] in NECESSARIOS
    ]

    if not gt_alvo:
        continue

    resultado = modelo.predict(
        source=str(caminho_imagem),
        conf=CONFIANCA_MINIMA,
        imgsz=IMGSZ,
        verbose=False,
    )[0]

    if resultado.boxes is None:
        continue

    previsoes = []

    for indice_pred, box in enumerate(resultado.boxes):
        classe_id = int(box.cls[0].item())
        classe = modelo.names[classe_id]

        if classe not in NECESSARIOS:
            continue

        confianca = float(box.conf[0].item())

        if confianca < CONFIANCA_MINIMA:
            continue

        x1, y1, x2, y2 = [
            float(valor)
            for valor in box.xyxy[0].tolist()
        ]

        previsoes.append(
            {
                "pred_id": indice_pred,
                "classe_id": classe_id,
                "classe": classe,
                "confianca": confianca,
                "xyxy": (x1, y1, x2, y2),
            }
        )

    if not previsoes:
        continue

    correspondencias = []

    for gt in gt_alvo:
        for pred in previsoes:
            if gt["classe"] != pred["classe"]:
                continue

            iou = calcular_iou(
                gt["xyxy"],
                pred["xyxy"],
            )

            if iou < IOU_MINIMO:
                continue

            correspondencias.append(
                {
                    "gt": gt,
                    "pred": pred,
                    "iou": iou,
                }
            )

    # Primeiro as melhores correspondências.
    correspondencias.sort(
        key=lambda item: (
            item["iou"],
            item["pred"]["confianca"],
        ),
        reverse=True,
    )

    gt_usados = set()
    pred_usados = set()

    for correspondencia in correspondencias:
        gt = correspondencia["gt"]
        pred = correspondencia["pred"]

        if gt["gt_id"] in gt_usados:
            continue

        if pred["pred_id"] in pred_usados:
            continue

        gt_usados.add(gt["gt_id"])
        pred_usados.add(pred["pred_id"])

        x1, y1, x2, y2 = pred["xyxy"]

        largura_box = x2 - x1
        altura_box = y2 - y1
        area_box = largura_box * altura_box
        centro_x = (x1 + x2) / 2

        posicao = classificar_posicao(
            centro_x=centro_x,
            largura_imagem=largura_imagem,
        )

        tamanho, ocupacao = classificar_tamanho(
            area_box=area_box,
            area_imagem=area_imagem,
        )

        posicao_vertical, proporcao_vertical = (
            classificar_vertical(
                y2=y2,
                altura_imagem=altura_imagem,
            )
        )

        gt_x1, gt_y1, gt_x2, gt_y2 = gt["xyxy"]

        candidatos_validos.append(
            {
                "arquivo": caminho_imagem.name,
                "caminho_imagem": str(caminho_imagem),
                "classe": pred["classe"],
                "confianca": pred["confianca"],
                "iou": correspondencia["iou"],
                "pred_x1": x1,
                "pred_y1": y1,
                "pred_x2": x2,
                "pred_y2": y2,
                "gt_x1": gt_x1,
                "gt_y1": gt_y1,
                "gt_x2": gt_x2,
                "gt_y2": gt_y2,
                "posicao": posicao,
                "tamanho": tamanho,
                "ocupacao_imagem": ocupacao,
                "posicao_vertical": posicao_vertical,
                "proporcao_vertical": proporcao_vertical,
            }
        )

df_todos_candidatos = pd.DataFrame(
    candidatos_validos
)

print()
print(
    "Correspondências válidas encontradas:",
    len(df_todos_candidatos),
)

if df_todos_candidatos.empty:
    raise RuntimeError(
        "Nenhuma correspondência válida foi encontrada."
    )

resumo_disponiveis = (
    df_todos_candidatos["classe"]
    .value_counts()
    .rename_axis("classe")
    .reset_index(name="quantidade")
)

display(resumo_disponiveis)

# 12. Selecionar candidatos complementares

A seleção usa uma regra fixa e reproduzível:

1. maior IoU;
2. maior confiança;
3. no máximo um candidato por imagem sempre que possível.

Não usamos o risco previsto por V1 ou V2 como critério.

In [ ]:
candidatos_selecionados = []
imagens_selecionadas = set()

for classe, meta in META_CANDIDATOS.items():
    df_classe = (
        df_todos_candidatos[
            df_todos_candidatos["classe"] == classe
        ]
        .copy()
        .sort_values(
            ["iou", "confianca"],
            ascending=[False, False],
        )
    )

    selecionados_classe = []

    # Primeira passagem:
    # tenta utilizar imagens distintas.
    for _, linha in df_classe.iterrows():
        if linha["arquivo"] in imagens_selecionadas:
            continue

        selecionados_classe.append(linha)
        imagens_selecionadas.add(linha["arquivo"])

        if len(selecionados_classe) >= meta:
            break

    # Segunda passagem:
    # se necessário, permite outra detecção de uma imagem já escolhida.
    if len(selecionados_classe) < meta:
        ids_existentes = {
            (
                item["arquivo"],
                item["pred_x1"],
                item["pred_y1"],
                item["pred_x2"],
                item["pred_y2"],
            )
            for item in selecionados_classe
        }

        for _, linha in df_classe.iterrows():
            identificador = (
                linha["arquivo"],
                linha["pred_x1"],
                linha["pred_y1"],
                linha["pred_x2"],
                linha["pred_y2"],
            )

            if identificador in ids_existentes:
                continue

            selecionados_classe.append(linha)
            ids_existentes.add(identificador)

            if len(selecionados_classe) >= meta:
                break

    candidatos_selecionados.extend(
        selecionados_classe
    )

    print(
        f"{classe}: "
        f"{len(selecionados_classe)} selecionados "
        f"de meta {meta}"
    )

df_candidatos = pd.DataFrame(
    candidatos_selecionados
).reset_index(drop=True)

df_candidatos.insert(
    0,
    "candidato_id",
    [
        f"R{i:03d}"
        for i in range(1, len(df_candidatos) + 1)
    ],
)

print()
print(
    "Total de candidatos complementares:",
    len(df_candidatos),
)

display(
    df_candidatos["classe"]
    .value_counts()
    .rename_axis("classe")
    .reset_index(name="quantidade")
)

## 13. Verificar se existe material suficiente

Se `wardrobe` tiver poucos candidatos corretos, isso não é um erro do notebook.

Pode indicar que o modelo tem dificuldade real nessa classe.

In [ ]:
print("Necessidade mínima para completar a validação:")
print(NECESSARIOS)
print()

contagem_selecionada = (
    df_candidatos["classe"]
    .value_counts()
    .to_dict()
)

insuficientes = []

for classe, minimo in NECESSARIOS.items():
    quantidade = contagem_selecionada.get(classe, 0)

    status = (
        "OK"
        if quantidade >= minimo
        else "INSUFICIENTE"
    )

    print(
        f"{classe:12s} "
        f"{quantidade:2d} candidatos | "
        f"mínimo necessário: {minimo:2d} | "
        f"{status}"
    )

    if quantidade < minimo:
        insuficientes.append(classe)

if insuficientes:
    print()
    print(
        "ATENÇÃO: não há candidatos suficientes "
        "para completar todas as classes:"
    )
    print(insuficientes)

# 14. Gerar imagens para revisão humana

A imagem de revisão mostra:

- a bounding box **prevista pelo YOLO**;
- o ID do candidato;
- a classe;
- IoU;
- confiança.

O risco V1/V2 não aparece.

Isso permite confirmar visualmente se o objeto é adequado para entrar no gabarito.

In [ ]:
def gerar_imagem_revisao(linha):
    imagem = Image.open(
        linha["caminho_imagem"]
    ).convert("RGB")

    desenho = ImageDraw.Draw(imagem)

    x1 = float(linha["pred_x1"])
    y1 = float(linha["pred_y1"])
    x2 = float(linha["pred_x2"])
    y2 = float(linha["pred_y2"])

    desenho.rectangle(
        [(x1, y1), (x2, y2)],
        width=5,
    )

    texto = (
        f'{linha["candidato_id"]} | '
        f'{linha["classe"]} | '
        f'IoU {linha["iou"]:.2f} | '
        f'Conf {linha["confianca"]:.2f}'
    )

    try:
        bbox_texto = desenho.textbbox(
            (0, 0),
            texto,
        )

        largura_texto = (
            bbox_texto[2] - bbox_texto[0]
        )

        altura_texto = (
            bbox_texto[3] - bbox_texto[1]
        )
    except Exception:
        largura_texto = len(texto) * 7
        altura_texto = 14

    x_texto = max(0, int(x1))
    y_texto = max(
        0,
        int(y1) - altura_texto - 8,
    )

    desenho.rectangle(
        [
            (x_texto, y_texto),
            (
                x_texto + largura_texto + 8,
                y_texto + altura_texto + 8,
            ),
        ],
        fill="black",
    )

    desenho.text(
        (x_texto + 4, y_texto + 4),
        texto,
        fill="white",
    )

    nome_classe = (
        linha["classe"]
        .replace(" ", "_")
    )

    destino = (
        PASTA_CANDIDATOS
        / f'{linha["candidato_id"]}_{nome_classe}.jpg'
    )

    imagem.save(
        destino,
        quality=95,
    )

    return destino


caminhos_revisao = []

for _, linha in df_candidatos.iterrows():
    caminhos_revisao.append(
        gerar_imagem_revisao(linha)
    )

df_candidatos["imagem_revisao"] = [
    str(caminho)
    for caminho in caminhos_revisao
]

print(
    f"{len(caminhos_revisao)} imagens salvas em:"
)
print(PASTA_CANDIDATOS)

# 15. Criar CSV para rotulagem manual

Serão adicionadas as colunas:

```text
risco_esperado
usar_no_gabarito
observacao
```

Na etapa manual:

- `risco_esperado`: `baixo`, `medio`, `alto` ou `excluir`;
- `usar_no_gabarito`: poderá ser preenchida depois com `sim` ou `nao`;
- `observacao`: justificativa opcional.

O arquivo não contém nenhuma previsão de risco V1/V2.

In [ ]:
df_saida = df_candidatos[
    [
        "candidato_id",
        "arquivo",
        "classe",
        "confianca",
        "iou",
        "posicao",
        "tamanho",
        "ocupacao_imagem",
        "posicao_vertical",
        "proporcao_vertical",
        "imagem_revisao",
    ]
].copy()

df_saida["risco_esperado"] = ""
df_saida["usar_no_gabarito"] = ""
df_saida["observacao"] = ""

if CAMINHO_CSV_CANDIDATOS.exists():
    print(
        "ATENÇÃO: o arquivo já existe."
    )
    print(
        "Ele NÃO será sobrescrito automaticamente "
        "para evitar perda de rotulagem manual."
    )
    print(CAMINHO_CSV_CANDIDATOS)
else:
    df_saida.to_csv(
        CAMINHO_CSV_CANDIDATOS,
        index=False,
        encoding="utf-8-sig",
    )

    print("CSV criado em:")
    print(CAMINHO_CSV_CANDIDATOS)

# ⛔ PARE AQUI

A primeira parte está concluída.

Agora faça o seguinte:

1. abra a pasta:

```text
reports/validation_risk/candidatos_complementares/
```

2. compacte a pasta em `.zip`;

3. localize:

```text
reports/tables/candidatos_complementares_risco.csv
```

4. registrar a revisão manual no ZIP e no CSV correspondentes.

## Não continue criando a validação final ainda

Primeiro precisamos selecionar:

```text
3 bed
5 sofa
8 wardrobe
```

que sejam visualmente válidos.

Depois esses 16 casos serão unidos aos 34 cenários válidos da primeira etapa,
formando o conjunto final de **50 cenários válidos**.

# 16. Informações de auditoria

Esta célula é opcional e serve para documentar como os candidatos foram encontrados.

In [ ]:
auditoria = pd.DataFrame(
    [
        {
            "parametro": "confianca_minima",
            "valor": CONFIANCA_MINIMA,
        },
        {
            "parametro": "iou_minimo",
            "valor": IOU_MINIMO,
        },
        {
            "parametro": "imagens_test_total",
            "valor": len(todas_imagens_test),
        },
        {
            "parametro": "imagens_ja_utilizadas",
            "valor": len(imagens_ja_utilizadas),
        },
        {
            "parametro": "imagens_disponiveis",
            "valor": len(imagens_disponiveis),
        },
        {
            "parametro": "correspondencias_validas",
            "valor": len(df_todos_candidatos),
        },
        {
            "parametro": "candidatos_selecionados",
            "valor": len(df_candidatos),
        },
    ]
)

display(auditoria)

caminho_auditoria = (
    PASTA_TABELAS
    / "auditoria_candidatos_complementares_risco.csv"
)

auditoria.to_csv(
    caminho_auditoria,
    index=False,
    encoding="utf-8-sig",
)

print("Auditoria salva em:")
print(caminho_auditoria)

# 17. Observação metodológica

Esta etapa foi desenhada para avaliar o módulo de risco **condicionado a uma detecção correta**.

Portanto:

```text
Ground truth
    ↓
objeto realmente existe
    ↓
YOLO detecta a mesma classe
    ↓
bounding boxes possuem IoU >= 0,50
    ↓
cenário pode ser enviado para avaliação de risco
```

Erros como:

```text
rack → wardrobe
poltrona → sofa
```

pertencem ao desempenho do **detector**, e não devem ser contabilizados como erros
da regra de classificação de risco.

Essa separação ajuda a manter a avaliação experimental mais clara:

- métricas YOLO → capacidade de detectar/localizar objetos;
- validação V1/V2 → capacidade de estimar o impacto do objeto na locomoção.